# Setting Up

## Importing raw data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Lat/lon into info

In [ ]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="sydney_demand_mapper")

def get_coords(place):
    """Return (lat, lon) for a suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None

# Apply geocoding to the 'Name' column
latitudes, longitudes = [], []
for suburb in info['Name']:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # polite pause to avoid hitting API limits

info['latitude'] = latitudes
info['longitude'] = longitudes

In [ ]:
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()
#Dee Why West doesn't exist as a suburb polygon, so will need to change the name to Dee Why

In [ ]:
dee_why_west_lat = -33.73441
dee_why_west_lon = 151.28278
#Found the lat/lon information online

In [ ]:
info.loc[info["Name"] == "Dee Why West", "latitude"] = dee_why_west_lat
info.loc[info["Name"] == "Dee Why West", "longitude"] = dee_why_west_lon
#inputting lat and lon from online into info

## Holiday Function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

#Monarch's Birthday
def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]   # Monday = 0
    return mondays[1]                   # second Monday



# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

holiday_order = list(HOLIDAYS_VIC.keys())

## Import CSV

In [ ]:
import pandas as pd

rank = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"
)


In [ ]:
rank.head()

## Time blocks

In [ ]:
blocks = {
    "00_04": range(0, 4),
    "04_10": range(4, 10),
    "10_15": range(10, 15),
    "15_20": range(15, 20),
    "20_24": range(20, 24)
}


## Temperature for plotting

In [ ]:
# Ensure datetime index
obs.index = pd.to_datetime(obs.index)

# Extract hour + date
obs["hour"] = obs.index.hour
obs["date"] = obs.index.date

# Prepare output list
temp_rows = []

# Loop over each day
for day, df_day in obs.groupby("date"):

    row = {"date": pd.Timestamp(day)}

    # Loop over each block
    for block_name, hours in blocks.items():
        block_vals = df_day[df_day["hour"].isin(hours)]["t2m"]

        # Mean temperature for this block
        row[f"{block_name}_temp_mean"] = block_vals.mean()

    temp_rows.append(row)

# Convert to DataFrame
temp_blocks = pd.DataFrame(temp_rows)


In [ ]:
# Ensure datetime
rank["date"] = pd.to_datetime(rank["date"])
temp_blocks["date"] = pd.to_datetime(temp_blocks["date"])

# Merge block-level temperatures into rank
rank = rank.merge(
    temp_blocks,
    on="date",
    how="left"
)


# 1x3 panel map (for each time block)
- public holiday, weekday and weekend
- ranking of the day minus weekend/day
- Difference in ranking for each time block
- mapped with substations above residential fraction of 75%

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

def map_ph_we_wd_block(
    df,
    info,
    holiday,
    block,
    df_station_col="station_name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean",
    crs_epsg=3857
):
    """
    Creates a 1×3 spatial map for a single time block:
        1. Mean Relative Rank
        2. Mean Relative Rank − Weekend Mean Relative Rank
        3. Mean Relative Rank − Weekday Mean Relative Rank
    """

    # 1. Filter to high-residential stations
    high_res = info[info[residential_col] >= residential_threshold].copy()
    high_res_names = high_res[info_station_col].unique()
    df = df[df[df_station_col].isin(high_res_names)].copy()

    # 2. Filter to selected holiday
    df_hol = df[df["holiday"] == holiday].copy()

    # 3. Prepare GeoDataFrame
    gdf = high_res.copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326"
    ).to_crs(crs_epsg)

    # 4. Compute PH, PH-WE, PH-WD
    results = []
    col = f"{block}{block_suffix}"

    for station in gdf[info_station_col]:
        df_s = df_hol[df_hol[df_station_col] == station]
        if df_s.empty:
            continue

        df_ph = df_s[df_s["is_holiday"] == True]
        ph_vals = df_ph.groupby("year")[col].mean().rename("ph")

        df_we = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == True)]
        we_vals = df_we.groupby("year")[col].mean().rename("we")

        df_wd = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == False)]
        wd_vals = df_wd.groupby("year")[col].mean().rename("wd")

        merged = (
            ph_vals.to_frame()
            .merge(we_vals, on="year", how="left")
            .merge(wd_vals, on="year", how="left")
        )

        merged["ph_we"] = merged["ph"] - merged["we"]
        merged["ph_wd"] = merged["ph"] - merged["wd"]

        vals = merged.mean()

        results.append({
            info_station_col: station,
            "ph": vals["ph"],
            "ph_we": vals["ph_we"],
            "ph_wd": vals["ph_wd"]
        })

    # 5. Merge results back into GeoDataFrame
    res_df = pd.DataFrame(results)
    gdf = gdf.merge(res_df, on=info_station_col, how="left")

    # 6. Compute padded extent for zoom-out
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 4800  # metres
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # 7. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

    # Overarching title using actual holiday name
    year_min = int(df_hol["year"].min())
    year_max = int(df_hol["year"].max())
    block_label = block.replace("_", ":").replace("15:20", "15:00–20:00")
    fig.suptitle(
        f"Relative Demand Patterns on {holiday} ({block_label}, 2-year rolling window ({year_min}–{year_max})\n"
        f"High‑residential locations only (≥ 0.75)",
        fontsize=16,
        y=1.02
    )



    panels = [
        ("Mean Relative Rank", "ph"),
        ("Mean Relative Rank − Weekend Mean Relative Rank", "ph_we"),
        ("Mean Relative Rank − Weekday Mean Relative Rank", "ph_wd")
    ]

    for ax, (title, colname) in zip(axes, panels):

        gdf.plot(
            ax=ax,
            column=colname,
            cmap="cool",
            legend=True,
            markersize=60,
            edgecolor="black",
            legend_kwds={
                "label": (
                    "Red = higher Public Holiday rank (higher relative demand)\n"
                    "Blue = lower Public Holiday rank"
                    if colname == "ph"
                    else "Red = Public Holiday > baseline, Blue = Public Holiday < baseline"
                )
            }
        )

        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])

        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

        for _, row in gdf.iterrows():
            ax.text(
                row.geometry.x + 800,
                row.geometry.y + 0,
                row[info_station_col],
                fontsize=7,
                ha="left",
                va="center"
            )

        ax.set_title(title)
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()


In [ ]:
map_ph_we_wd_block(
    df=rank,
    info=info,
    holiday="Monarch's Birthday",
    block="15_20"
)


In [ ]:
print(rank["station_code"].unique()[:20])
print(info["energy_asset"].unique()[:20])


In [ ]:
print(rank["station_name"].unique()[:20])
print(info["Name"].unique()[:20])


## Producing a table of the mapped results

In [ ]:
def make_ph_we_wd_table(
    df,
    info,
    holiday,
    block,
    df_station_col="station_name",
    info_station_col="Name",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean"
):
    """
    Returns a table of:
        - <holiday> Mean Relative Rank
        - <holiday> − Weekend Mean Relative Rank
        - <holiday> − Weekday Mean Relative Rank
        - Residential Fraction
    """

    # Filter to high-residential stations
    high_res = info[info[residential_col] >= residential_threshold].copy()
    high_res_names = high_res[info_station_col].unique()
    df = df[df[df_station_col].isin(high_res_names)].copy()

    # Filter to selected holiday
    df_hol = df[df["holiday"] == holiday].copy()

    results = []
    col = f"{block}{block_suffix}"

    for station in high_res_names:
        df_s = df_hol[df_hol[df_station_col] == station]
        if df_s.empty:
            continue

        # PH
        df_ph = df_s[df_s["is_holiday"] == True]
        ph_vals = df_ph.groupby("year")[col].mean().rename("ph")

        # Weekend baseline
        df_we = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == True)]
        we_vals = df_we.groupby("year")[col].mean().rename("we")

        # Weekday baseline
        df_wd = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == False)]
        wd_vals = df_wd.groupby("year")[col].mean().rename("wd")

        merged = (
            ph_vals.to_frame()
            .merge(we_vals, on="year", how="left")
            .merge(wd_vals, on="year", how="left")
        )

        merged["ph_we"] = merged["ph"] - merged["we"]
        merged["ph_wd"] = merged["ph"] - merged["wd"]

        vals = merged.mean()

        results.append({
            info_station_col: station,
            "ph": vals["ph"],
            "ph_we": vals["ph_we"],
            "ph_wd": vals["ph_wd"]
        })

    table = pd.DataFrame(results)

    # Merge residential fraction from info
    table = table.merge(
        info[[info_station_col, residential_col]],
        on=info_station_col,
        how="left"
    )

    # Rename columns using the actual holiday name
    table = table.rename(columns={
        "ph": f"{holiday} Mean Relative Rank",
        "ph_we": f"{holiday} − Weekend Mean Relative Rank",
        "ph_wd": f"{holiday} − Weekday Mean Relative Rank",
        residential_col: "Residential Fraction"
    })

    # Sort for readability
    table = table.sort_values(f"{holiday} Mean Relative Rank", ascending=False).reset_index(drop=True)

    return table


In [ ]:
table = make_ph_we_wd_table(rank, info, "Monarch's Birthday", "15_20")
table


# Mapping and saving
- using the function above
- adding in looping for all public holidays
- saving maps to '/home/565/pv3484/aus_substation_electricity/figures/relative_demand_difference_weekend_weekday'
- the mean of all 2‑year windows across the entire 2004–2017 period
- not each window separately

## New holiday function
- clumps together all easter related holidays and christmas + boxing day

In [ ]:
HOLIDAY_GROUPS = {
    "Good Friday": "Easter Long Weekend",
    "Easter Saturday": "Easter Long Weekend",
    "Easter Sunday": "Easter Long Weekend",
    "Easter Monday": "Easter Long Weekend",

    "Christmas Day": "Christmas and Boxing Day",
    "Boxing Day": "Christmas and Boxing Day",
}


## Combined-holiday mapping function
Core function. It averages PH, PH–WE, PH–WD across multiple holidays and plots one map.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

def map_ph_we_wd_block_combined(
    df,
    info,
    holidays,
    holiday_group_name,     # NEW: the display name, e.g. "Easter Long Weekend"
    block,
    df_station_col="station_name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean",
    crs_epsg=3857
):

    # Block labels for titles
    BLOCK_LABELS = {
        "00_04": "00:00–04:00",
        "04_10": "04:00–10:00",
        "10_15": "10:00–15:00",
        "15_20": "15:00–20:00",
        "20_24": "20:00–00:00"
    }
    block_label = BLOCK_LABELS.get(block, block)

    # 1. Filter to high-residential stations
    high_res = info[info[residential_col] >= residential_threshold].copy()
    high_res_names = high_res[info_station_col].unique()
    df = df[df[df_station_col].isin(high_res_names)].copy()

    # 2. Filter to selected holidays (list)
    df_hol = df[df["holiday"].isin(holidays)].copy()

    # 3. Prepare GeoDataFrame
    gdf = high_res.copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326"
    ).to_crs(crs_epsg)

    # 4. Compute PH, PH-WE, PH-WD averaged across holidays
    results = []
    col = f"{block}{block_suffix}"

    for station in gdf[info_station_col]:
        df_s = df_hol[df_hol[df_station_col] == station]
        if df_s.empty:
            continue

        df_ph = df_s[df_s["is_holiday"] == True]
        ph_vals = (
            df_ph.groupby(["holiday", "year"])[col]
            .mean()
            .groupby("year")
            .mean()
            .rename("ph")
        )

        df_we = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == True)]
        we_vals = df_we.groupby("year")[col].mean().rename("we")

        df_wd = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == False)]
        wd_vals = df_wd.groupby("year")[col].mean().rename("wd")

        merged = (
            ph_vals.to_frame()
            .merge(we_vals, on="year", how="left")
            .merge(wd_vals, on="year", how="left")
        )

        merged["ph_we"] = merged["ph"] - merged["we"]
        merged["ph_wd"] = merged["ph"] - merged["wd"]

        vals = merged.mean()

        results.append({
            info_station_col: station,
            "ph": vals["ph"],
            "ph_we": vals["ph_we"],
            "ph_wd": vals["ph_wd"]
        })

    # 5. Merge results back into GeoDataFrame
    res_df = pd.DataFrame(results)
    gdf = gdf.merge(res_df, on=info_station_col, how="left")

    # 6. Compute padded extent for zoom-out
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 4800
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # 7. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

    year_min = int(df_hol["year"].min())
    year_max = int(df_hol["year"].max())

    fig.suptitle(
        f"Relative Demand Patterns ({holiday_group_name}) ({block_label}, 2-year rolling window {year_min}–{year_max})\n"
        f"High‑residential locations only (≥ 0.75)",
        fontsize=16,
        y=1.05
    )

    panels = [
        (f"{holiday_group_name} mean relative rank", "ph"),
        ("Mean Relative Rank − Weekend Mean Relative Rank", "ph_we"),
        ("Mean Relative Rank − Weekday Mean Relative Rank", "ph_wd")
    ]

    for ax, (title, colname) in zip(axes, panels):

        gdf.plot(
            ax=ax,
            column=colname,
            cmap="cool",
            legend=True,
            markersize=60,
            edgecolor="black",
            legend_kwds={
                "label": "Blue = Public Holiday < baseline, Pink = Public Holiday > baseline"
            }
        )

        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])

        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

        for _, row in gdf.iterrows():
            ax.text(
                row.geometry.x + 800,
                row.geometry.y,
                row[info_station_col],
                fontsize=7,
                ha="left",
                va="center"
            )

        ax.set_title(title)
        ax.set_axis_off()

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)

    return fig


## Batch function
This groups holidays, averages them, and saves one map per block.

In [ ]:
def generate_all_relative_demand_maps(
    df,
    info,
    blocks,
    output_root="/home/565/pv3484/aus_substation_electricity/figures/relative_demand_difference_weekend_weekday"
):

    holiday_groups = {
        "Good Friday": "Easter Long Weekend",
        "Easter Saturday": "Easter Long Weekend",
        "Easter Sunday": "Easter Long Weekend",
        "Easter Monday": "Easter Long Weekend",
        "Christmas Day": "Christmas and Boxing Day",
        "Boxing Day": "Christmas and Boxing Day",
    }

    def group_holiday_name(h):
        return holiday_groups.get(h, h)

    os.makedirs(output_root, exist_ok=True)

    all_holidays = sorted(df["holiday"].unique())

    grouped = {}
    for h in all_holidays:
        g = group_holiday_name(h)
        grouped.setdefault(g, []).append(h)

    for grouped_name, holiday_list in grouped.items():

        safe_group = grouped_name.replace(" ", "_")
        holiday_dir = os.path.join(output_root, safe_group)
        os.makedirs(holiday_dir, exist_ok=True)

        for block in blocks:

            fig = map_ph_we_wd_block_combined(
                df=df,
                info=info,
                holidays=holiday_list,
                holiday_group_name=grouped_name,
                block=block
            )

            filename = f"{safe_group}_{block}.png"
            filepath = os.path.join(holiday_dir, filename)
            fig.savefig(filepath, dpi=300, bbox_inches="tight")
            plt.close(fig)


## Runs

In [ ]:
blocks = ["00_04", "04_10", "10_15", "15_20", "20_24"]
generate_all_relative_demand_maps(rank, info, blocks)


# Different mapping
- separate maps for each 2‑year rolling window by building a new
- clean set of functions that do NOT touch your existing ones

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

def map_ph_we_wd_block_window(
    df,
    info,
    holidays,
    holiday_group_name,
    block,
    year,                      # NEW: the holiday year, representing window year–year+1
    df_station_col="station_name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean",
    crs_epsg=3857
):

    BLOCK_LABELS = {
        "00_04": "00:00–04:00",
        "04_10": "04:00–10:00",
        "10_15": "10:00–15:00",
        "15_20": "15:00–20:00",
        "20_24": "20:00–00:00"
    }
    block_label = BLOCK_LABELS.get(block, block)

    # Filter to high-residential stations
    high_res = info[info[residential_col] >= residential_threshold].copy()
    high_res_names = high_res[info_station_col].unique()

    # Filter to this rolling window (year Y = window Y–Y+1)
    df = df[df[df_station_col].isin(high_res_names)]
    df = df[df["year"] == year]
    df = df[df["holiday"].isin(holidays)]

    # Prepare GeoDataFrame
    gdf = high_res.copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326"
    ).to_crs(crs_epsg)

    # Compute PH, PH-WE, PH-WD
    results = []
    col = f"{block}{block_suffix}"

    for station in gdf[info_station_col]:
        df_s = df[df[df_station_col] == station]
        if df_s.empty:
            continue

        df_ph = df_s[df_s["is_holiday"] == True]
        ph_vals = df_ph[col].mean()

        df_we = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == True)]
        we_vals = df_we[col].mean()

        df_wd = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == False)]
        wd_vals = df_wd[col].mean()

        results.append({
            info_station_col: station,
            "ph": ph_vals,
            "ph_we": ph_vals - we_vals if we_vals is not None else None,
            "ph_wd": ph_vals - wd_vals if wd_vals is not None else None
        })

    # Merge results
    res_df = pd.DataFrame(results)
    gdf = gdf.merge(res_df, on=info_station_col, how="left")

    # Extent
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 4800
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

    fig.suptitle(
        f"{holiday_group_name} ({block_label}) — Rolling window {year}–{year+1}\n"
        f"High‑residential locations only (≥ 0.75)",
        fontsize=16,
        y=1.05
    )

    panels = [
        (f"{holiday_group_name} mean relative rank", "ph"),
        ("Mean Relative Rank − Weekend Mean Relative Rank", "ph_we"),
        ("Mean Relative Rank − Weekday Mean Relative Rank", "ph_wd")
    ]

    for ax, (title, colname) in zip(axes, panels):

        gdf.plot(
            ax=ax,
            column=colname,
            cmap="cool",
            legend=True,
            markersize=60,
            edgecolor="black",
            legend_kwds={
                "label": "Blue = Public Holiday < baseline, Pink = Public Holiday > baseline"
            }
        )

        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])

        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

        for _, row in gdf.iterrows():
            ax.text(
                row.geometry.x + 800,
                row.geometry.y,
                row[info_station_col],
                fontsize=7,
                ha="left",
                va="center"
            )

        ax.set_title(title)
        ax.set_axis_off()

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)

    return fig


In [ ]:
import os

def generate_window_maps(
    df,
    info,
    blocks,
    output_root="/home/565/pv3484/aus_substation_electricity/figures/relative_demand_difference_year_windows"
):

    # Holiday grouping
    holiday_groups = {
        "Good Friday": "Easter Long Weekend",
        "Easter Saturday": "Easter Long Weekend",
        "Easter Sunday": "Easter Long Weekend",
        "Easter Monday": "Easter Long Weekend",
        "Christmas Day": "Christmas and Boxing Day",
        "Boxing Day": "Christmas and Boxing Day",
    }

    def group_name(h):
        return holiday_groups.get(h, h)

    os.makedirs(output_root, exist_ok=True)

    # Build holiday → group mapping
    all_holidays = sorted(df["holiday"].unique())
    grouped = {}
    for h in all_holidays:
        g = group_name(h)
        grouped.setdefault(g, []).append(h)

    # Each year corresponds to a rolling window year–year+1
    years = sorted(df["year"].unique())

    for group, holiday_list in grouped.items():

        safe_group = group.replace(" ", "_")
        group_dir = os.path.join(output_root, safe_group)
        os.makedirs(group_dir, exist_ok=True)

        for year in years:

            # Folder for this rolling window
            window_dir = os.path.join(group_dir, f"{year}_{year+1}")
            os.makedirs(window_dir, exist_ok=True)

            for block in blocks:

                fig = map_ph_we_wd_block_window(
                    df=df,
                    info=info,
                    holidays=holiday_list,
                    holiday_group_name=group,
                    block=block,
                    year=year
                )

                filename = f"{block}.png"
                fig.savefig(os.path.join(window_dir, filename), dpi=300, bbox_inches="tight")
                plt.close(fig)


In [ ]:
blocks = ["00_04", "04_10", "10_15", "15_20", "20_24"]
generate_window_maps(rank, info, blocks)


# Fixing plots from Ailie's suggestions
- making the first map absolute values
- the second the third plot are sequential but the y axis are on the same scale
- Remove Umina location (improved eligibility)

## Building a custom colour bar for public holiday bin

In [ ]:
import matplotlib
cube = matplotlib.colormaps.get_cmap("cubehelix")
ph_colors = [cube(i/9) for i in range(10)]


In [ ]:
from matplotlib.colors import ListedColormap
ph_cmap = ListedColormap(ph_colors)

## Plotting

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import matplotlib
from matplotlib.colors import ListedColormap

def map_ph_we_wd_block_combined_v7(
    df,
    info,
    holidays,
    holiday_group_name,
    block,
    df_station_col="Name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean_relative_rank",
    crs_epsg=3857
):
    """
    v7:
    - Larger maps
    - Reduced spacing between panels
    - Shared colourbar placed BELOW panels 2 & 3
    - PH renamed to Public Holiday everywhere
    - PH binned legend placed left-middle of Panel 1
    - Uses 'Name' as the join key
    """

    # -----------------------------
    # 1. Block label
    # -----------------------------
    BLOCK_LABELS = {
        "00_04": "00:00–04:00",
        "04_10": "04:00–10:00",
        "10_15": "10:00–15:00",
        "15_20": "15:00–20:00",
        "20_24": "20:00–00:00"
    }
    block_label = BLOCK_LABELS.get(block, block)

    # -----------------------------
    # 2. Filter high-residential stations
    # -----------------------------
    high_res = info[info[residential_col] >= residential_threshold].copy()
    high_res = high_res[high_res[info_station_col] != "Umina"].copy()
    high_res_names = high_res[info_station_col].unique()

    df = df[df[df_station_col].isin(high_res_names)].copy()
    df = df[df[df_station_col] != "Umina"].copy()

    # -----------------------------
    # 3. Filter to selected holidays
    # -----------------------------
    df_hol = df[df["holiday_group"].isin(holidays)].copy()

    # -----------------------------
    # 4. Build GeoDataFrame
    # -----------------------------
    gdf = high_res.copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326"
    ).to_crs(crs_epsg)

    # -----------------------------
    # 5. Compute Public Holiday metrics
    # -----------------------------
    results = []
    col = f"{block}{block_suffix}"

    for station in gdf[info_station_col]:
        df_s = df_hol[df_hol[df_station_col] == station]
        if df_s.empty:
            continue

        df_ph = df_s[df_s["is_holiday"] == True]
        ph_vals = (
            df_ph.groupby(["holiday_group", "year"])[col]
            .mean()
            .groupby("year")
            .mean()
            .rename("ph")
        )

        df_we = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == True)]
        we_vals = df_we.groupby("year")[col].mean().rename("we")

        df_wd = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == False)]
        wd_vals = df_wd.groupby("year")[col].mean().rename("wd")

        merged = (
            ph_vals.to_frame()
            .merge(we_vals, on="year", how="left")
            .merge(wd_vals, on="year", how="left")
        )

        merged["ph_we"] = merged["ph"] - merged["we"]
        merged["ph_wd"] = merged["ph"] - merged["wd"]

        vals = merged.mean()

        results.append({
            info_station_col: station,
            "ph": vals["ph"],
            "ph_we": vals["ph_we"],
            "ph_wd": vals["ph_wd"]
        })

    # -----------------------------
    # 6. Merge results into GeoDataFrame
    # -----------------------------
    res_df = pd.DataFrame(results)
    gdf = gdf.merge(res_df, on=info_station_col, how="left")

    # -----------------------------
    # 7. Map extent padding
    # -----------------------------
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 4800
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # -----------------------------
    # 8. Fixed PH bins
    # -----------------------------
    bins = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]
    labels = [f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins)-1)]

    gdf["ph_bin"] = pd.cut(
        gdf["ph"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # -----------------------------
    # 9. Plasma palette for PH bins
    # -----------------------------
    plasma = matplotlib.colormaps.get_cmap("plasma")
    positions = np.linspace(0, 1, 7) ** 0.7
    ph_colors = [plasma(p) for p in positions]
    ph_cmap = ListedColormap(ph_colors)

    # -----------------------------
    # 10. Shared scale for difference panels
    # -----------------------------
    diff_min = min(gdf["ph_we"].min(), gdf["ph_wd"].min())
    diff_max = max(gdf["ph_we"].max(), gdf["ph_wd"].max())

    # -----------------------------
    # 11. Plotting
    # -----------------------------
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    for ax in axes:
        ax.set_aspect("equal", adjustable="box")

    year_min = int(df_hol["year"].min())
    year_max = int(df_hol["year"].max())

    fig.suptitle(
        f"Relative Demand Patterns ({holiday_group_name}) "
        f"({block_label}, 2-year rolling window {year_min}–{year_max})\n"
        f"High-residential locations only (≥ 0.75)",
        fontsize=17,
        y=0.999
    )

    # After creating fig, axes
    title_y = 0.99  # or 0.985 if you want them tighter
    axes[0].set_title("Public Holiday Relative Rank (binned)", y=title_y)
    axes[1].set_title("Public Holiday − Weekend Mean Relative Rank", y=title_y)
    axes[2].set_title("Public Holiday − Weekday Mean Relative Rank", y=title_y)


    # -----------------------------
    # Panel 1 — Public Holiday binned
    # -----------------------------
    gdf.plot(
        ax=axes[0],
        column="ph_bin",
        cmap=ph_cmap,
        markersize=70,
        edgecolor="black",
        legend=False
    )

    handles = [
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=ph_colors[i], markersize=10)
        for i in range(len(labels))
    ]

    fig.legend(
        handles,
        labels,
        title="Public Holiday Relative Rank (binned)",
        loc="center left",
        bbox_to_anchor=(0.005, 0.50),
        frameon=False
    )

    # -----------------------------
    # Panels 2 & 3 — shared scale
    # -----------------------------
    panel_info = [
        ("Public Holiday − Weekend Mean Relative Rank", "ph_we"),
        ("Public Holiday − Weekday Mean Relative Rank", "ph_wd")
    ]

    for ax, (title, colname) in zip(axes[1:], panel_info):
        gdf.plot(
            ax=ax,
            column=colname,
            cmap="inferno",
            vmin=diff_min,
            vmax=diff_max,
            markersize=70,
            edgecolor="black",
            legend=False
        )
        # no ax.set_title here

    # -----------------------------
    # Shared colourbar BELOW panels 2 & 3
    # -----------------------------
    sm = plt.cm.ScalarMappable(
        cmap="inferno",
        norm=plt.Normalize(vmin=diff_min, vmax=diff_max)
    )
    sm._A = []

    cbar = fig.colorbar(
        sm,
        ax=axes[1:],
        orientation="horizontal",
        fraction=0.05,
        pad=0.20
    )
    cbar.set_label(f"Public Holiday difference scale ({diff_min:.2f} to {diff_max:.2f})")

    # -----------------------------
    # Basemap + labels
    # -----------------------------
    for ax in axes:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

        for _, row in gdf.iterrows():
            ax.text(
                row.geometry.x + 800,
                row.geometry.y,
                row[info_station_col],
                fontsize=8,
                ha="left",
                va="center"
            )

        ax.set_axis_off()

    # -----------------------------
    # Final spacing
    # -----------------------------
    plt.subplots_adjust(
        top=0.86,
        bottom=0.16,
        wspace=0.03,
        hspace=0.01
    )

    return fig

In [ ]:
# Example: test the new function for New Year's Day
fig = map_ph_we_wd_block_combined_v7(
    df=rank,
    info=info,
    holidays=["New Year's Day"],
    holiday_group_name="New Year's Day",
    block="10_15",
    df_station_col="Name",
    info_station_col="Name",
    block_suffix="_mean_relative_rank"
)

plt.show()


## Trying to fix misshing Monarch's Birthday

In [ ]:
def map_ph_we_wd_block_combined_v8(
    df,
    info,
    holidays,
    holiday_group_name,
    block,
    df_station_col="Name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean_relative_rank",
    crs_epsg=3857
):
    """
    v8:
    - IDENTICAL visual style to v7
    - Fixed station set across all holidays:
        Residential >= 0.75, not 'Umina', AND has data for ANY holiday
    - Stations with missing PH/WE/WD for this holiday/block are plotted in light grey
    - 'No data' added to legend
    """

    import pandas as pd
    import geopandas as gpd
    import matplotlib.pyplot as plt
    import contextily as ctx
    import numpy as np
    import matplotlib
    from matplotlib.colors import ListedColormap

    # -----------------------------
    # 1. Block label
    # -----------------------------
    BLOCK_LABELS = {
        "00_04": "00:00–04:00",
        "04_10": "04:00–10:00",
        "10_15": "10:00–15:00",
        "15_20": "15:00–20:00",
        "20_24": "20:00–00:00"
    }
    block_label = BLOCK_LABELS.get(block, block)

    # -----------------------------
    # 2. Determine globally eligible stations
    # -----------------------------
    high_res = info[
        (info[residential_col] >= residential_threshold)
        & (info[info_station_col] != "Umina")
    ][info_station_col]

    block_cols = [
        f"{b}{block_suffix}"
        for b in ["00_04", "04_10", "10_15", "15_20", "20_24"]
    ]

    stations_with_data = (
        df.groupby(df_station_col)[block_cols]
        .apply(lambda x: x.notna().any().any())
    )
    stations_with_data = stations_with_data[stations_with_data].index

    eligible_stations = set(high_res).intersection(stations_with_data)

    # -----------------------------
    # 3. Filter df to eligible stations
    # -----------------------------
    df = df[df[df_station_col].isin(eligible_stations)].copy()

    # -----------------------------
    # 4. Filter to selected holidays
    # -----------------------------
    df_hol = df[df["holiday_group"].isin(holidays)].copy()

    # -----------------------------
    # 5. Build GeoDataFrame
    # -----------------------------
    gdf = info[info[info_station_col].isin(eligible_stations)].copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326"
    ).to_crs(crs_epsg)

    # -----------------------------
    # 6. Compute PH, WE, WD metrics
    # -----------------------------
    results = []
    col = f"{block}{block_suffix}"

    for station in eligible_stations:
        df_s = df_hol[df_hol[df_station_col] == station]

        if df_s.empty:
            ph_val = np.nan
            ph_we_val = np.nan
            ph_wd_val = np.nan
        else:
            df_ph = df_s[df_s["is_holiday"] == True]
            ph_vals = (
                df_ph.groupby(["holiday_group", "year"])[col]
                .mean()
                .groupby("year")
                .mean()
                .rename("ph")
            )

            df_we = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == True)]
            we_vals = df_we.groupby("year")[col].mean().rename("we")

            df_wd = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == False)]
            wd_vals = df_wd.groupby("year")[col].mean().rename("wd")

            merged = (
                ph_vals.to_frame()
                .merge(we_vals, on="year", how="left")
                .merge(wd_vals, on="year", how="left")
            )

            merged["ph_we"] = merged["ph"] - merged["we"]
            merged["ph_wd"] = merged["ph"] - merged["wd"]

            vals = merged.mean()

            ph_val = vals["ph"]
            ph_we_val = vals["ph_we"]
            ph_wd_val = vals["ph_wd"]

        results.append({
            info_station_col: station,
            "ph": ph_val,
            "ph_we": ph_we_val,
            "ph_wd": ph_wd_val
        })

    res_df = pd.DataFrame(results)
    gdf = gdf.merge(res_df, on=info_station_col, how="left")

    # -----------------------------
    # 7. Map extent padding
    # -----------------------------
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 4800
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # -----------------------------
    # 8. Fixed PH bins
    # -----------------------------
    bins = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]
    labels = [f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins)-1)]

    gdf["ph_bin"] = pd.cut(
        gdf["ph"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # -----------------------------
    # 9. Plasma palette for PH bins
    # -----------------------------
    plasma = matplotlib.colormaps.get_cmap("plasma")
    positions = np.linspace(0, 1, 7) ** 0.7
    ph_colors = [plasma(p) for p in positions]
    ph_cmap = ListedColormap(ph_colors)

    # -----------------------------
    # 10. Shared scale for difference panels
    # -----------------------------
    diff_min = min(gdf["ph_we"].min(), gdf["ph_wd"].min())
    diff_max = max(gdf["ph_we"].max(), gdf["ph_wd"].max())

    # -----------------------------
    # 11. Plotting
    # -----------------------------
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    for ax in axes:
        ax.set_aspect("equal", adjustable="box")

    year_min = int(df_hol["year"].min())
    year_max = int(df_hol["year"].max())

    fig.suptitle(
        f"Relative Demand Patterns ({holiday_group_name}) "
        f"({block_label}, 2-year rolling window {year_min}–{year_max})\n"
        f"High-residential locations only (≥ 0.75)",
        fontsize=17,
        y=0.999
    )

    title_y = 0.99
    axes[0].set_title("Public Holiday Relative Rank (binned)", y=title_y)
    axes[1].set_title("Public Holiday − Weekend Mean Relative Rank", y=title_y)
    axes[2].set_title("Public Holiday − Weekday Mean Relative Rank", y=title_y)

    # -----------------------------
    # Panel 1 — PH binned
    # -----------------------------
    gdf.plot(
        ax=axes[0],
        column="ph_bin",
        cmap=ph_cmap,
        markersize=70,
        edgecolor="black",
        legend=False,
        missing_kwds={"color": "#e6e6e6", "label": "No data"}
    )

    handles = [
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=ph_colors[i], markersize=10)
        for i in range(len(labels))
    ]
    handles.append(
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor="#e6e6e6", markersize=10,
                   label="No data")
    )

    fig.legend(
        handles,
        labels + ["No data"],
        title="Public Holiday Relative Rank (binned)",
        loc="center left",
        bbox_to_anchor=(0.005, 0.50),
        frameon=False
    )

    # -----------------------------
    # Panels 2 & 3 — shared scale
    # -----------------------------
    panel_info = [
        ("Public Holiday − Weekend Mean Relative Rank", "ph_we"),
        ("Public Holiday − Weekday Mean Relative Rank", "ph_wd")
    ]

    for ax, (title, colname) in zip(axes[1:], panel_info):
        gdf.plot(
            ax=ax,
            column=colname,
            cmap="inferno",
            vmin=diff_min,
            vmax=diff_max,
            markersize=70,
            edgecolor="black",
            legend=False,
            missing_kwds={"color": "#e6e6e6", "label": "No data"}
        )

    sm = plt.cm.ScalarMappable(
        cmap="inferno",
        norm=plt.Normalize(vmin=diff_min, vmax=diff_max)
    )
    sm._A = []

    cbar = fig.colorbar(
        sm,
        ax=axes[1:],
        orientation="horizontal",
        fraction=0.05,
        pad=0.20
    )
    cbar.set_label(f"Public Holiday difference scale ({diff_min:.2f} to {diff_max:.2f})")

    # -----------------------------
    # Basemap + labels
    # -----------------------------
    for ax in axes:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

        for _, row in gdf.iterrows():
            ax.text(
                row.geometry.x + 800,
                row.geometry.y,
                row[info_station_col],
                fontsize=8,
                ha="left",
                va="center"
            )

        ax.set_axis_off()

    plt.subplots_adjust(
        top=0.86,
        bottom=0.16,
        wspace=0.03,
        hspace=0.01
    )

    return fig


In [ ]:
# Example: test the new function for New Year's Day
fig = map_ph_we_wd_block_combined_v8(
    df=rank,
    info=info,
    holidays=["Australia Day"],
    holiday_group_name="Australia Day",
    block="10_15",
    df_station_col="Name",
    info_station_col="Name",
    block_suffix="_mean_relative_rank"
)

plt.show()


## Looping through all and saving v7

In [ ]:
import os

def save_all_holiday_block_maps(
    df,
    info,
    holidays,
    blocks,
    base_dir="/g/data/ng72/pv3484/substation_data/figures/mrr_pb_minus_we_wd"
):
    """
    For each holiday:
        create folder <base_dir>/<holiday_name>/
        for each block:
            generate map
            save as <block>.png
    """

    # Ensure base directory exists
    os.makedirs(base_dir, exist_ok=True)

    for hol in holidays:

        # Create a safe folder name
        safe_hol = hol.replace(" ", "_")
        hol_dir = os.path.join(base_dir, safe_hol)
        os.makedirs(hol_dir, exist_ok=True)

        print(f"\n=== {hol} ===")
        print(f"Saving maps to: {hol_dir}")

        for block in blocks:

            # Generate figure
            fig = map_ph_we_wd_block_combined_v7(
                df=df,
                info=info,
                holidays=[hol],
                holiday_group_name=hol,
                block=block
            )

            # Build filename
            outpath = os.path.join(hol_dir, f"{block}.png")

            # Save
            fig.savefig(outpath, dpi=300, bbox_inches="tight")
            plt.close(fig)

            print(f"  Saved: {outpath}")


In [ ]:
sorted(info["Name"].unique())[:20]

In [ ]:
set(rank[rank["holiday_group"] == "Monarch's Birthday"]["Name"]) - set(info[info["Residential"] >= 0.75]["Name"])


In [ ]:
all_holidays = sorted(rank["holiday_group"].unique())

all_blocks = ["00_04", "04_10", "10_15", "15_20", "20_24"]

save_all_holiday_block_maps(
    df=rank,
    info=info,
    holidays=all_holidays,
    blocks=all_blocks
)
